# Data checking and cleaning guide
# 資料檢查與清理指南

Example dataset / 範例資料: Hugging Face engine data (19,535 rows, 6 sensors, target `Engine Condition`). Change two settings in section 0 to use it on AI4I.
範例是 Hugging Face 的引擎資料。改第 0 節的兩個設定，就能套用到 AI4I。

**Rules / 原則**
1. Never edit the raw file. Work on a copy. / 不改原始檔，只改複本。
2. Look first, record every decision, change data last. / 先看、再記錄、最後才改。
3. Never delete rows silently: count them and say why. / 不要默默刪資料：先算幾筆、說明原因。

> Chart labels are in English because Colab's default chart font cannot show Chinese.
> 圖表標籤使用英文，因為 Colab 預設的圖表字型無法顯示中文。

## 0. Settings and load / 設定與載入

**To use AI4I / 改用 AI4I:** set `SOURCE = "upload"`, upload `ai4i2020.csv`, then set
`TARGET = "Machine failure"` and `DROP_COLS = ["UDI", "TWF", "HDF", "PWF", "OSF", "RNF"]`.
The five failure-type columns must be dropped: they are the answer in disguise (data leakage).
五個故障類型欄位必須排除，它們等於把答案放進特徵（資料洩漏）。

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOURCE = "url"      # "url" or "upload" / "url" 或 "upload"
URL = "https://huggingface.co/datasets/jskswamy/predictive-maintenance-data/resolve/main/engine_data.csv"
TARGET = "Engine Condition"
DROP_COLS = []      # columns that must NOT be features / 不可當特徵的欄位

if SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(up.values()))))
else:
    df = pd.read_csv(URL)

raw = df.copy()     # untouched copy / 原始複本
print("Loaded / 已載入:", df.shape)
df.head()

## 1. First look and integrity / 初步檢查與完整性

Check size, types, missing cells, duplicate rows and the target balance.
檢查大小、型態、缺值、重複列與目標欄位的比例。

In [ ]:
print("shape / 形狀:", df.shape)
print("\ndtypes / 型態:\n", df.dtypes)
print("\nmissing per column / 各欄缺值:\n", df.isna().sum())
print("\nduplicate rows / 重複列:", int(df.duplicated().sum()))
print("\ntarget counts / 目標數量:\n", df[TARGET].value_counts())
print("\ntarget share / 目標比例:\n", df[TARGET].value_counts(normalize=True).round(3))

## 2. Ranges and zeros / 數值範圍與零值

Read the min and max of every column and ask: is this physically possible? Many zeros may mean a sensor that read nothing.
看每一欄的最小與最大值，問自己：這在物理上可能嗎？大量的 0 可能代表感測器沒讀到。

In [ ]:
num_cols = [c for c in df.columns
            if c != TARGET and c not in DROP_COLS and pd.api.types.is_numeric_dtype(df[c])]
print("Feature columns / 特徵欄位:", num_cols)

display(df[num_cols].describe().T.round(2))

zeros = (df[num_cols] == 0).sum()
display(pd.DataFrame({"zeros": zeros, "zero_share": (zeros / len(df)).round(3)}))

## 3. Outliers / 異常值

The IQR rule only flags values worth a look. A flagged value is **not** automatically an error.
IQR 規則只是標出「值得看一眼」的值，被標出的不一定是錯誤。

The last column shows the average target among the flagged rows. Compare it with the overall average: if they differ a lot, the outliers carry information and deleting them may hurt.
最後一欄是被標出的列的目標平均值。跟整體平均比較：差很多代表異常值有資訊，刪掉可能有害。

In [ ]:
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    m = (df[c] < lo) | (df[c] > hi)
    rows.append([c, lo, hi, int(m.sum()), m.mean() * 100,
                 df.loc[m, TARGET].mean() if m.any() else np.nan])
outl = pd.DataFrame(rows, columns=["column", "low_limit", "high_limit", "n_outliers",
                                   "pct_of_rows", "target_mean_in_outliers"]).round(3)
display(outl)
print("Overall target mean / 整體目標平均:", round(df[TARGET].mean(), 3))

In [ ]:
n = len(num_cols)
rows_ = int(np.ceil(n / 3))
fig, axes = plt.subplots(rows_, 3, figsize=(14, 3.6 * rows_))
for ax, c in zip(np.ravel(axes), num_cols):
    for v, g in df.groupby(TARGET):
        ax.hist(g[c].dropna(), bins=40, alpha=.5, label=f"{TARGET}={v}")
    ax.set_title(c)
np.ravel(axes)[0].legend()
plt.tight_layout(); plt.show()

## 4. Does each feature relate to the target? / 每個特徵跟目標有關嗎？

Average of each feature by target value, plus how well each single feature separates the target (AUC; 0.5 = no signal). A single feature with AUC near 1.0 is a warning sign of leakage.
依目標值看各特徵的平均，以及單一特徵區分目標的能力（AUC，0.5 代表沒有訊號）。單一特徵的 AUC 接近 1.0 是資料洩漏的警訊。

In [ ]:
from sklearn.metrics import roc_auc_score

display(df.groupby(TARGET)[num_cols].mean().T.round(2))

auc = {}
for c in num_cols:
    s = df[c].fillna(df[c].median())
    a = roc_auc_score(df[TARGET], s)
    auc[c] = max(a, 1 - a)          # direction does not matter / 方向不重要
display(pd.Series(auc, name="single_feature_AUC").sort_values(ascending=False).round(3))

corr = df[num_cols].corr()
plt.figure(figsize=(6, 5)); plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(n), num_cols, rotation=90); plt.yticks(range(n), num_cols)
plt.colorbar(label="correlation"); plt.title("Feature correlation"); plt.tight_layout(); plt.show()

## 5. Decide and record / 決定並記錄

Fill the log below before changing anything. Options for a suspicious value: **keep**, **flag** (add a 0/1 column), **cap** (limit to a range) or **remove**. Prefer flag or keep unless you can show it is a data error.
改任何東西之前先填記錄。對可疑的值：**保留**、**標記**（新增 0/1 欄）、**限制範圍**或**移除**。除非能證明是資料錯誤，優先選標記或保留。

| Finding / 發現 | Rows / 筆數 | Decision / 決定 | Reason / 理由 |
|---|---|---|---|
| e.g. Coolant Temp above 100 °C | ? | ? | ? |
| e.g. Pressure equal to 0 | ? | ? | ? |
| Duplicate rows | ? | ? | ? |
| Missing cells | ? | ? | ? |

In [ ]:
# Example only: FLAG suspicious rows instead of deleting them. Edit the column and limit.
# 範例：用「標記」代替刪除。請依你的欄位和門檻修改。
COL, LIMIT = None, None            # e.g. COL, LIMIT = "Coolant Temp", 100
df = raw.copy()
if COL:
    df["flag_" + COL.replace(" ", "_")] = (df[COL] > LIMIT).astype(int)
    print("Flagged rows / 被標記的列:", int((df[COL] > LIMIT).sum()), "of", len(df))
df = df.drop_duplicates()
print("After dropping exact duplicates / 移除完全重複列後:", df.shape)

## 6. Split correctly / 正確切分

Stratified split keeps the target share the same. Also check that identical rows do not sit in both train and test.
分層切分讓訓練與測試的目標比例一致。也要確認相同的列沒有同時出現在訓練與測試。

If rows come in time order or in runs of near-identical rows (like the hydraulic data), a random split is too optimistic. Use blocked groups instead (see the hydraulic notebook).
若資料有時間順序或連續相似（像液壓系統），隨機切分會太樂觀，應改用區塊切分（見液壓系統 notebook）。

In [ ]:
from sklearn.model_selection import train_test_split

feat_cols = [c for c in df.columns if c != TARGET and c not in DROP_COLS
             and pd.api.types.is_numeric_dtype(df[c])]
X, y = df[feat_cols], df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("target share train / test:", round(y_tr.mean(), 3), "/", round(y_te.mean(), 3))
same = pd.merge(X_tr.drop_duplicates(), X_te.drop_duplicates(), how="inner").shape[0]
print("identical feature rows in both train and test / 同時在訓練與測試的相同列:", same)

## 7. Baseline before anything fancy / 先做基準

A model must beat "always predict the majority class". Use PR-AUC, precision, recall and F1, not only accuracy.
模型至少要贏過「永遠猜最多的類別」。除了 accuracy，也看 PR-AUC、precision、recall、F1。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, average_precision_score,
                             precision_score, recall_score, f1_score)

majority = max(y_te.mean(), 1 - y_te.mean())
print("Accuracy of always guessing the majority class / 永遠猜多數類別的 accuracy:", round(majority, 3))

models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1),
}
rows = []
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]; pred = (p >= 0.5).astype(int)
    rows.append([name, accuracy_score(y_te, pred), average_precision_score(y_te, p),
                 precision_score(y_te, pred), recall_score(y_te, pred), f1_score(y_te, pred)])
pd.DataFrame(rows, columns=["model", "accuracy", "PR-AUC", "precision", "recall", "F1"]).round(3)

## 8. Checklist before you trust the result / 相信結果之前的檢查表

- [ ] Raw file untouched; work done on a copy / 原始檔未改動
- [ ] Missing cells and duplicates counted and handled / 缺值與重複已統計並處理
- [ ] Every odd value looked at and logged (keep, flag, cap or remove) / 每個可疑值都看過並記錄
- [ ] No column leaks the answer (single-feature AUC near 1.0?) / 沒有欄位洩漏答案
- [ ] Split is stratified; no identical rows on both sides / 切分分層，且兩邊沒有相同列
- [ ] Baseline beaten, and not only on accuracy / 贏過基準，而且不只看 accuracy
- [ ] Source and license of the data written down / 資料的來源與授權已記下
- [ ] If provenance is unknown (as with this engine data), say so in the report / 來源不明時，報告中要明說